# 📱 Mobile Price Classification

**Obiettivo:** aiutare Bob, fondatore di una nuova azienda di cellulari, a capire
quali caratteristiche tecniche determinano il prezzo di vendita e a prevedere
la **fascia di prezzo** (0 = economico … 3 = premium) di un telefono.

**Dataset:** [Mobile Price Classification — Kaggle](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification)

- `train.csv` — 2.000 righe con target `price_range` (usato per addestramento e validazione)
- `test.csv` — 1.000 righe senza target (per l'eventuale submission Kaggle)

**Pipeline:** EDA → correlazioni → baseline Decision Tree → Random Forest → valutazione → feature importance

## 1. Setup e caricamento dati

Per riprodurre: scaricare i due CSV dal link Kaggle qui sopra e metterli nella
stessa cartella del notebook.

In [1]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# grafici compatti: notebook piu' leggero, anteprima comunque nitida
matplotlib.rcParams["figure.dpi"] = 70
matplotlib.rcParams["savefig.bbox"] = "tight"

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

print("Train:", df_train.shape, "| Test:", df_test.shape)
print("Colonna presente solo nel train (il target):",
      set(df_train.columns) - set(df_test.columns))

Train: (2000, 21) | Test: (1000, 21)
Colonna presente solo nel train (il target): {'price_range'}


## 2. Primo contatto con i dati

Comandi standard appena si carica un dataset: dimensioni, prime righe,
tipi di dato e statistiche di base.

In [2]:
print(df_train.head())
print()
df_train.info()

   battery_power  blue  clock_speed  dual_sim  fc  four_g  int_memory  m_dep  \
0            842     0          2.2         0   1       0           7    0.6   
1           1021     1          0.5         1   0       1          53    0.7   
2            563     1          0.5         1   2       1          41    0.9   
3            615     1          2.5         0   0       0          10    0.8   
4           1821     1          1.2         0  13       1          44    0.6   

   mobile_wt  n_cores  ...  px_height  px_width   ram  sc_h  sc_w  talk_time  \
0        188        2  ...         20       756  2549     9     7         19   
1        136        3  ...        905      1988  2631    17     3          7   
2        145        5  ...       1263      1716  2603    11     2          9   
3        131        6  ...       1216      1786  2769    16     8         11   
4        141        2  ...       1208      1212  1411     8     2         15   

   three_g  touch_screen  wifi  price_

In [3]:
df_train.describe().round(2)

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
count,2000.00,2000.0,2000.00,2000.00,2000.00,2000.00,2000.00,2000.00,2000.00,2000.00,...,2000.00,2000.00,2000.00,2000.00,2000.00,2000.00,2000.00,2000.0,2000.00,2000.00
mean,1238.52,0.5,1.52,0.51,4.31,0.52,32.05,0.50,140.25,4.52,...,645.11,1251.52,2124.21,12.31,5.77,11.01,0.76,0.5,0.51,1.50
std,439.42,0.5,0.82,0.50,4.34,0.50,18.15,0.29,35.40,2.29,...,443.78,432.20,1084.73,4.21,4.36,5.46,0.43,0.5,0.50,1.12
min,501.00,0.0,0.50,0.00,0.00,0.00,2.00,0.10,80.00,1.00,...,0.00,500.00,256.00,5.00,0.00,2.00,0.00,0.0,0.00,0.00
25%,851.75,0.0,0.70,0.00,1.00,0.00,16.00,0.20,109.00,3.00,...,282.75,874.75,1207.50,9.00,2.00,6.00,1.00,0.0,0.00,0.75
50%,1226.00,0.0,1.50,1.00,3.00,1.00,32.00,0.50,141.00,4.00,...,564.00,1247.00,2146.50,12.00,5.00,11.00,1.00,1.0,1.00,1.50
75%,1615.25,1.0,2.20,1.00,7.00,1.00,48.00,0.80,170.00,7.00,...,947.25,1633.00,3064.50,16.00,9.00,16.00,1.00,1.0,1.00,2.25
max,1998.00,1.0,3.00,1.00,19.00,1.00,64.00,1.00,200.00,8.00,...,1960.00,1998.00,3998.00,19.00,18.00,20.00,1.00,1.0,1.00,3.00


**Osservazioni:**
- ✅ Nessun valore mancante (tutte le colonne sono `non-null` al 100%)
- ✅ Tutte le colonne sono numeriche (`int64` / `float64`): nessuna conversione necessaria
- Le colonne `blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen`, `wifi` sono **binarie** (0/1):
  la loro media coincide con la quota di telefoni dotati di quella caratteristica

## 3. Distribuzione del target

In [4]:
df_train["price_range"].value_counts().sort_index()

price_range
0    500
1    500
2    500
3    500
Name: count, dtype: int64

**Target perfettamente bilanciato**: 500 telefoni per ciascuna delle 4 fasce.
Nessun problema di classi sbilanciate.

## 4. Correlazione con il prezzo

Quanto ogni caratteristica "si muove insieme" a `price_range` (da -1 a +1).

In [5]:
corr = (df_train.corr()["price_range"]
        .drop("price_range")
        .sort_values(ascending=False))
corr.round(3)

ram              0.917
battery_power    0.201
px_width         0.166
px_height        0.149
int_memory       0.044
sc_w             0.039
pc               0.034
three_g          0.024
sc_h             0.023
fc               0.022
talk_time        0.022
blue             0.021
wifi             0.019
dual_sim         0.017
four_g           0.015
n_cores          0.004
m_dep            0.001
clock_speed     -0.007
mobile_wt       -0.030
touch_screen    -0.030
Name: price_range, dtype: float64

**La RAM domina** (0.917). Seguono `battery_power` (0.20) e la risoluzione
dello schermo (~0.15). Tutto il resto e' rumore (< 0.05).

In [6]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(data=df_train, x="price_range", y="ram", ax=ax,
            hue="price_range", palette="viridis", legend=False)
ax.set_title("RAM per fascia di prezzo", fontsize=14, fontweight="bold")
ax.set_xlabel("Fascia di prezzo (0 = economico, 3 = premium)")
ax.set_ylabel("RAM (MB)")
plt.show()

Le scatole quasi **non si sovrappongono**: la RAM da sola separa quasi
perfettamente le 4 fasce. Ottimo presagio per il machine learning.

## 5. Machine Learning

### Preparazione X / y

> ⚠️ **Regola anti data-leakage:** `y` contiene il target; da `X` si toglie
> **solo** il target. Togliere feature (es. la RAM) distrugge il modello:
> senza RAM l'accuracy crolla da 83% a 24.7%, il livello del caso puro.

In [7]:
from sklearn.model_selection import train_test_split

y = df_train["price_range"]
X = df_train.drop(columns=["price_range"])

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", x_train.shape, "| Validation:", x_test.shape)

Train: (1600, 20) | Validation: (400, 20)


### Baseline: Decision Tree

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

tree = DecisionTreeClassifier(random_state=42)
tree.fit(x_train, y_train)
acc_tree = accuracy_score(y_test, tree.predict(x_test))
print(f"Accuracy Decision Tree: {acc_tree:.3f}")

Accuracy Decision Tree: 0.833


### Random Forest

Stesso identico pattern sklearn (crea → fit → predict → valuta): cambiano
solo import e costruttore.

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
acc_rf = accuracy_score(y_test, y_pred)
print(f"Accuracy Random Forest: {acc_rf:.3f}")

Accuracy Random Forest: 0.892


## 6. Matrice di confusione

L'accuracy dice *quante* volte il modello sbaglia; la matrice dice **dove**.

In [10]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
ax.set_title("Matrice di confusione - Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Previsione del modello")
ax.set_ylabel("Risposta vera")
plt.show()

**Tutti gli errori sono tra fasce adiacenti** (mai confusione 0↔3): il modello
non commette errori commercialmente gravi. L'errore piu' frequente e' la sottostima
di un gradino dei telefoni premium (15 casi 3→2).

## 7. Feature importance

Quanto pesa ogni caratteristica nelle decisioni del modello (somma = 100%).

In [11]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
imp.plot.barh(ax=ax, color="#2c7fb8")
ax.set_title("Feature Importance - Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Peso nella decisione (somma = 1.0)")
for i, v in enumerate(imp):
    ax.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=8)
plt.show()

## 8. Conclusioni per Bob

1. **La RAM determina quasi la meta' delle decisioni** (47% di peso, correlazione 0.92):
   e' la leva di prezzo principale.
2. **Batteria e risoluzione schermo** formano la seconda linea (~19% combinato).
3. **Le feature binarie** (bluetooth, wifi, 4G, dual SIM...) pesano meno dell'1%
   ciascuna: sono attese dal mercato, non giustificano un prezzo piu' alto.
4. Il modello finale (Random Forest) raggiunge l'**89% di accuracy** e non confonde
   mai fasce lontane: e' affidabile per stimare il posizionamento di prezzo
   di un nuovo modello.

In [1]:
import pandas as pd
import sqlite3 
from pathlib import Path

BASE_DIR = Path(".")

df = pd.read_csv(BASE_DIR / "train.csv")

(BASE_DIR / "data").mkdir(exist_ok=True)

conn= sqlite3.connect(BASE_DIR/"data"/"dati.db")
df.to_sql("mobile", conn, if_exists="replace", index=False)

check=pd.read_sql("SELECT COUNT(*) FROM mobile", conn)
print(check)
conn.close()


   COUNT(*)
0      2000


2026-10-25 16:29:07.306 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.311 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.317 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.323 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.326 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.330 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.333 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:29:07.338 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()